[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/notebooks/03_gaussian_pi_pulse.ipynb)

**Run the _Setup_ cell below first**, then run the remaining cells top-to-bottom. The `assert` statements are the tutorial's built-in checks — if every cell runs without error, every step passed.

> _Auto-generated from [`docs/tutorials/03_gaussian_pi_pulse.md`](https://github.com/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/03_gaussian_pi_pulse.md) by `tools/build_tutorial_notebooks.py`. Edit the Markdown tutorial, not this notebook._

In [ ]:
# Setup — install iontrap-dynamics and its dependencies (qutip, numpy, scipy).
# First run on Colab takes ~1-2 min; safe to re-run (a no-op once installed).
%pip install -q "iontrap-dynamics[plot] @ git+https://github.com/uwarring82/iontrap-dynamics.git@main"

# Tutorial 3 — Gaussian π-pulse with `modulated_carrier_hamiltonian`


**Goal.** Keep the four-step skeleton from
[Tutorial 1](https://uwarring82.github.io/iontrap-dynamics/tutorials/01_first_rabi_readout/) and
[Tutorial 2](https://uwarring82.github.io/iontrap-dynamics/tutorials/02_red_sideband_fock1/) — configure → build → solve
→ read out — and swap the static carrier Hamiltonian for a
**time-dependent** one whose instantaneous Rabi frequency
`Ω(t) = Ω · f(t)` is shaped by a Gaussian envelope. By the end
you'll have delivered a clean `|↓⟩ → |↑⟩` π-rotation by choosing
the envelope amplitude so that the **pulse area** integrates to
exactly π, and you'll have seen the Bloch vector trace out a
smooth meridian in the y–z plane (not a bang-bang square-wave
trajectory).

The reference script is [`tools/run_demo_gaussian_pulse.py`][demo] —
the same scenario, used as the first Phase 1 end-to-end public-API
exercise. Its committed output bundle under
[`benchmarks/data/gaussian_pi_pulse_demo/`][bundle] includes the
plot embedded below.

[demo]: https://github.com/uwarring82/iontrap-dynamics/blob/main/tools/run_demo_gaussian_pulse.py
[bundle]: https://github.com/uwarring82/iontrap-dynamics/tree/main/benchmarks/data/gaussian_pi_pulse_demo

**Expected time.** ~10 min reading; ~1 s runtime.

**Level.** `core` — assumes the basics (Tutorials 0–1).

**Prerequisites.** [Tutorial 1](https://uwarring82.github.io/iontrap-dynamics/tutorials/01_first_rabi_readout/) — the
four-step skeleton. Nothing from Tutorial 2 is needed; the pulse
stays on the carrier (no sideband physics). Passing familiarity
with the interaction-picture carrier Hamiltonian at the level of
[`CONVENTIONS.md`](https://uwarring82.github.io/iontrap-dynamics/conventions/) §9.

---

> 📝 **Note** — New here? Read this first
>
>
> - Tutorials 1–2 drove the spin with a *constant* carrier; here the drive amplitude follows a shaped envelope `f(t)` — a smooth Gaussian bump that ramps the laser on and back off.
> - What sets the spin rotation is the **pulse area** `θ = ∫ Ω·f(t) dt`, *not* the peak height — you normalise the envelope so the area comes out to exactly `π`.
> - Area `π` means a `π`-pulse: it carries `|↓⟩` all the way to `|↑⟩` (a half-turn of the Bloch vector).
> - A time-dependent Hamiltonian is handed to `solve()` in the `[operator, coefficient-function]` list form, not as one fixed operator — the builder does that wrapping for you.
> - With zero laser phase the Bloch vector traces a clean meridian in the `y–z` plane: `⟨σ_x⟩ = 0`, `⟨σ_y⟩ = sin θ`, `⟨σ_z⟩ = −cos θ`.
> - Same four-step skeleton as Tutorials 1–2 (configure → build → solve → read out); only the Hamiltonian builder changes to `modulated_carrier_hamiltonian`.
> - **In a hurry?** Step 2 builds the envelope and normalises its area to `π`; Step 3 runs the shaped pulse and traces the meridian — that pair is the core.

**Symbols in this tutorial**

| Symbol | Plain meaning |
| --- | --- |
| `Ω` | reference (full-drive) Rabi rate — the rate at `f = 1`; the area-normalised envelope peaks at `A ≈ 0.4`, so the instantaneous `Ω(t)` never actually reaches `Ω`. |
| `f(t)` | dimensionless envelope shape (a Gaussian bump); multiplies `Ω`. |
| `Ω(t) = Ω·f(t)` | instantaneous Rabi rate the spin actually feels at time `t`. |
| `θ(t) = ∫₀^t Ω·f dt′` | accumulated pulse area — the rotation angle reached so far. |
| `σ` | Gaussian pulse width — tall-and-narrow (small `σ`) vs short-and-wide (large `σ`). |
| `A` | envelope amplitude, fixed so the *total* area is exactly `π`. |
| `π`-pulse | a pulse of total area `π`; rotates the spin \|↓⟩ → \|↑⟩. |

## The scenario

One ²⁵Mg⁺ ion, same trap as Tutorials 1–2 (axial mode at
`ω_mode / 2π = 1.5 MHz`, but the carrier-only dynamics don't touch
it — the mode slot is present only so `HilbertSpace` has something
to attach to). The laser is tuned on resonance at carrier Rabi
frequency `Ω / 2π = 1 MHz` — the same value as Tutorial 1.
Starting state: `|↓, 0⟩`.

What's different: the laser intensity is **gated** by a Gaussian
envelope centred at `t_c = 2.5 μs` with `σ = 0.5 μs`, over a total
pulse window `T = 5 μs`. The instantaneous Rabi frequency is

```
Ω(t) = Ω · f(t),    f(t) = A · exp(−(t − t_c)² / (2σ²))
```

and the envelope amplitude `A` is chosen so that the pulse area

```
θ(T) = ∫₀^T Ω · f(t) dt = π
```

comes out to exactly π — a clean single π-rotation. For a Gaussian
well inside the window (σ ≪ t_c, T − t_c), the integral collapses
to `Ω · A · σ · √(2π)`, so

```
A = π / (Ω · σ · √(2π))
```

Plugging in `Ω / 2π = 1 MHz`, `σ = 0.5 μs` gives `A ≈ 0.399`. The
Bloch trajectory during the pulse is the integrated rotation:

```
⟨σ_x⟩(t) = 0                    (phase φ = 0)
⟨σ_y⟩(t) = sin(θ(t))            θ(t) = ∫₀^t Ω · f(t') dt'
⟨σ_z⟩(t) = −cos(θ(t))
```

so the state smoothly traces a meridian in the y–z plane from the
south pole (`⟨σ_z⟩ = −1`) at `t = 0` to the north pole
(`⟨σ_z⟩ = +1`) at `t = T`.

## Step 1 — Configure (same three objects as Tutorial 1)

Identical to Tutorial 1's Step 1 — the pulse shape is a property of
the *Hamiltonian builder*, not of the `DriveConfig`. The
`carrier_rabi_frequency_rad_s` field stays the **peak-envelope**
Rabi frequency Ω; the envelope scales it from there.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from iontrap_dynamics.drives import DriveConfig
from iontrap_dynamics.modes import ModeConfig
from iontrap_dynamics.species import mg25_plus
from iontrap_dynamics.system import IonSystem

# House colours — match the reference figure above.
BLUE, RED, GREEN, PURPLE, GREY = "#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#444444"

# --- Boilerplate: same hardware as Tutorials 1–2 (species, axial mode, geometry).
#     None of these lines shape the pulse — the envelope arrives in Step 2.
mode = ModeConfig(
    label="axial",
    frequency_rad_s=2 * np.pi * 1.5e6,
    eigenvector_per_ion=np.array([[0.0, 0.0, 1.0]]),
)
system = IonSystem.homogeneous(species=mg25_plus(), n_ions=1, modes=(mode,))

# --- The physics that matters here: the drive carries the reference (full-drive) Ω;
#     the Gaussian envelope (Step 2) scales it in time (its peak is only ~0.4·Ω).
drive = DriveConfig(
    k_vector_m_inv=[0.0, 0.0, 2 * np.pi / 280e-9],
    carrier_rabi_frequency_rad_s=2 * np.pi * 1.0e6,  # Ω/2π = 1 MHz
    phase_rad=0.0,
)

> ⚠️ **Warning** — Common confusion — `f(t)` is a shape, `Ω` is the rate
>
>
> Keep `carrier_rabi_frequency_rad_s` equal to the **reference** (full-drive,
> `f = 1`) Rabi rate `Ω` and let the dimensionless envelope `f(t)` carry the
> time profile — the two multiply to give `Ω(t) = Ω·f(t)`. Do *not*
> fold the envelope's peak height into `Ω`: the amplitude `A` you
> compute in Step 2 is a normalisation of the *shape* fixed by the
> area target, not a second rate. If a pulse rotates by the wrong
> angle, first check which of the two you changed.

## Step 2 — Define the envelope and build the Hamiltonian

`modulated_carrier_hamiltonian` is the generic pulse-envelope
primitive. It takes a Python callable `envelope: Callable[[float], float]`
and returns the Hamiltonian in QuTiP's **time-dependent list format**
`[[H_carrier, coeff_fn]]` — not a bare `Qobj` like the static
builders. You don't need to care about the internal wrapping;
`sequences.solve` dispatches both formats transparently.

In [ ]:
import math

from iontrap_dynamics.hamiltonians import modulated_carrier_hamiltonian
from iontrap_dynamics.hilbert import HilbertSpace

hilbert = HilbertSpace(system=system, fock_truncations={"axial": 3})

# Pulse parameters
pulse_duration_s = 5.0e-6
pulse_centre_s = pulse_duration_s / 2.0
pulse_sigma_s = pulse_duration_s / 10.0  # = 0.5 μs

# Pulse-area normalisation — area(Gaussian · Ω) = π
rabi_rad_s = drive.carrier_rabi_frequency_rad_s
envelope_amplitude = np.pi / (rabi_rad_s * pulse_sigma_s * np.sqrt(2 * np.pi))

print(f"Step 2 — Ω/2π = {rabi_rad_s / (2 * np.pi) / 1e6:.1f} MHz,  "
      f"σ = {pulse_sigma_s * 1e6:.1f} μs,  T = {pulse_duration_s * 1e6:.1f} μs")
print(f"Step 2 — envelope amplitude A = {envelope_amplitude:.4f}  "
      f"(area = A·Ω·σ·√2π = π ✓)")

def gaussian_envelope(t: float) -> float:
    return envelope_amplitude * math.exp(
        -((t - pulse_centre_s) ** 2) / (2 * pulse_sigma_s**2)
    )

# Plot the envelope shape so readers see the Gaussian gate before the dynamics.
t_env = np.linspace(0.0, pulse_duration_s, 400)
f_env = np.array([gaussian_envelope(t) for t in t_env])

fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(t_env * 1e6, f_env, color=BLUE)
ax.axhline(envelope_amplitude, color=GREY, linewidth=0.8, linestyle="--", label=f"peak A = {envelope_amplitude:.4f}")
ax.set_xlabel("time  (μs)")
ax.set_ylabel("envelope  f(t) = A · exp(…)")
ax.set_title("Gaussian pulse envelope")
ax.legend(frameon=False)
plt.show()

hamiltonian = modulated_carrier_hamiltonian(
    hilbert, drive, ion_index=0, envelope=gaussian_envelope
)

> 📝 **Note** — Why the envelope is a plain `Callable[[float], float]`
>
>
> Many QuTiP time-dependent-list coefficients are written as
> `(t, args) -> float` closures keyed on a QuTiP-specific `args`
> dict. The builder lifts that boilerplate for you: pass a pure
> Python function of `t` (SI seconds), and the wrapping into
> QuTiP's step-callback is handled internally. The envelope must
> be deterministic — the solver samples it at every sub-step,
> and stochasticity there would invalidate the integrator's
> step-size control.

> 💡 **Tip** — Choosing `pulse_sigma` and the window
>
>
> Keep the Gaussian well inside `[0, T]` — the pulse-area formula
> `∫Ω·f = Ω·A·σ·√(2π)` uses the Gaussian's full-plane integral,
> and the window-truncation error is the erf-complement of the
> clipped tails. For `σ = T/10` centred at `T/2` the truncation
> is ~1e−30 (negligible); dropping to `σ = T/4` brings it up to
> ~1e−2, and you'd need to either widen the window or replace
> the analytic pulse-area formula with a numerical
> `scipy.integrate.quad` over the finite window.

> ⚠️ **Warning** — Common confusion — it's the area, not the peak
>
>
> The rotation angle is the *integral* `θ = ∫Ω·f dt`, so any two
> pulses with the same area rotate the spin by the same amount: a
> tall-narrow Gaussian and a short-wide one both make a `π`-pulse.
> That is exactly why `A ≈ 0.399` here even though a naive "amplitude
> `1`" guess would over-rotate. If you change `σ`, you must rescale
> `A` to hold the area at `π` — halve `σ` and you double `A`.

## Step 3 — Solve with three Bloch-component observables

The on-resonance carrier with zero laser phase drives the spin
along the y axis in the `{σ_x, σ_y, σ_z}` basis. `⟨σ_x⟩` stays at
zero for the whole pulse; `⟨σ_y⟩` and `⟨σ_z⟩` together trace out
the rotation. Requesting all three in one `solve` call gives you
the full Bloch trajectory to compare against analytics.

In [ ]:
from iontrap_dynamics.observables import spin_x, spin_y, spin_z
from iontrap_dynamics.sequences import solve
from iontrap_dynamics.states import ground_state

psi_0 = ground_state(hilbert)
times = np.linspace(0.0, pulse_duration_s, 400)

result = solve(
    hilbert=hilbert,
    hamiltonian=hamiltonian,
    initial_state=psi_0,
    times=times,
    observables=[
        spin_x(hilbert, 0),
        spin_y(hilbert, 0),
        spin_z(hilbert, 0),
    ],
)

sigma_x = result.expectations["sigma_x_0"]
sigma_y = result.expectations["sigma_y_0"]
sigma_z = result.expectations["sigma_z_0"]

The analytic prediction is the integrated rotation angle
`θ(t) = ∫₀^t Ω · f(t') dt'`, which you can evaluate with a
cumulative trapezoidal rule on a finer grid than the solver output:

In [ ]:
fine = np.linspace(times[0], times[-1], 20 * len(times))
omega_f_fine = rabi_rad_s * np.array([gaussian_envelope(t) for t in fine])
theta_fine = np.concatenate(
    ([0.0], np.cumsum(0.5 * (omega_f_fine[:-1] + omega_f_fine[1:])) * np.diff(fine)[0])
)
theta = np.interp(times, fine, theta_fine)

max_error = max(
    np.max(np.abs(sigma_x)),                    # analytic σ_x ≡ 0
    np.max(np.abs(sigma_y - np.sin(theta))),
    np.max(np.abs(sigma_z + np.cos(theta))),
)
print(f"Step 3 — final pulse area θ(T) = {theta[-1]:.7f} rad  (target π = {np.pi:.7f})")
print(f"Step 3 — max |⟨σᵢ⟩_numeric − analytic| = {max_error:.2e}")
print(f"Step 3 — final ⟨σ_z⟩ = {float(sigma_z[-1]):.8f}  (target +1)")
assert max_error < 1e-5, "numeric Bloch trajectory must match analytic (0, sin θ, −cos θ) to 1e-5; a larger gap points at the envelope, the area normalisation, or the basis — not at the shaped-pulse physics"
assert abs(theta[-1] - np.pi) < 1e-5, "accumulated area θ(T) must land on π; this checks the classical envelope integral (the A normalisation), independent of the quantum solve"   # total pulse area = π
assert sigma_z[-1] > 0.9999, "solved final ⟨σ_z⟩ must sit within 1e-4 of the north pole +1 — a clean π-rotation |↓⟩ → |↑⟩"             # ended at the north pole

# Plot the Bloch-vector trajectory against the analytic prediction.
t_us = times * 1e6
fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(t_us, np.sin(theta),    color=GREY,   linewidth=1.0, label=r"analytic $\sin\theta$")
ax.plot(t_us, -np.cos(theta),   color=GREY,   linewidth=1.0, linestyle="--",
        label=r"analytic $-\cos\theta$")
ax.scatter(t_us[::5], np.array(sigma_y)[::5], color=GREEN,  s=12, zorder=3,
           label=r"numeric $\langle\sigma_y\rangle$")
ax.scatter(t_us[::5], np.array(sigma_z)[::5], color=BLUE,   s=12, zorder=3,
           label=r"numeric $\langle\sigma_z\rangle$")
ax.scatter(t_us[::5], np.array(sigma_x)[::5], color=RED,    s=10, marker="s", zorder=3,
           label=r"numeric $\langle\sigma_x\rangle \approx 0$")
ax.set_xlabel("time  (μs)")
ax.set_ylabel("Bloch component")
ax.set_title("Bloch trajectory — Gaussian π-pulse")
ax.legend(frameon=False, fontsize=7)
plt.show()

**Takeaway.** `⟨σ_y⟩` rises to `+1` and falls back to `0` while `⟨σ_z⟩` sweeps `−1 → +1`: the Bloch vector walks a single clean meridian, crossing the equator (`θ = π/2`) exactly when half the pulse area has accumulated — for a symmetric Gaussian that is the pulse centre `t = 2.5 μs`.

Three independent assertions: the numerical Bloch trajectory
matches the analytic `(0, sin θ, −cos θ)` curve to better than
`1e-5`, the integrated pulse area comes out to π, and the final
spin projection is essentially `+1` (a clean π-rotation).

## Step 4 — Read out (still the same `SpinReadout`)

The readout layer is agnostic to how the trajectory was produced —
static carrier, sideband flop, or shaped pulse all feed the same
`SpinReadout` / `DetectorConfig` pair:

In [ ]:
from iontrap_dynamics import DetectorConfig, SpinReadout

detector = DetectorConfig(efficiency=0.5, dark_count_rate=0.3, threshold=3)
readout = SpinReadout(
    ion_index=0, detector=detector, lambda_bright=20.0, lambda_dark=0.0
)
measurement = readout.run(result, shots=500, seed=20260421)
bright_fraction = measurement.sampled_outcome["spin_readout_bright_fraction"]
print(f"Step 4 — final bright fraction = {float(bright_fraction[-1]):.4f}  "
      f"(500 shots, efficiency 0.5, dark-count rate 0.3)")

# Plot the sampled bright fraction across all time steps, mirroring the
# spin-up population that rises from 0 to ~1 during the π-pulse.
fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(times * 1e6, bright_fraction, color=BLUE, linewidth=1.0, label="sampled bright fraction")
ax.set_xlabel("time  (μs)")
ax.set_ylabel("bright fraction  (500 shots)")
ax.set_title("Step 4 — detector readout trajectory")
ax.legend(frameon=False)
plt.show()

At the final time step the bright fraction should land near the
expected `0.5 · (1 + 0.9999) ≈ 0.9999` brightening probability
(modulo detector inefficiency and dark counts, both absorbed into
the `DetectorConfig` model from Tutorial 1).

## Putting it together

The committed reference run (`Ω/2π = 1 MHz`, `σ = 0.5 μs`, 400
time points, `N_Fock = 3`) produces:

![Gaussian π-pulse](https://raw.githubusercontent.com/uwarring82/iontrap-dynamics/main/benchmarks/data/gaussian_pi_pulse_demo/plot.png)

Top panel: the Gaussian envelope `f(t)` — peaks at `t = 2.5 μs`,
amplitude `A ≈ 0.399`. Below it: the three Bloch components.
`⟨σ_x⟩` stays pinned at 0. `⟨σ_y⟩` rises, peaks at `sin(π/2) = 1`
around `t = 2.5 μs` (the half-area point), then decays back to 0
as the pulse tail fills in the second half-rotation. `⟨σ_z⟩`
swings cleanly from `−1` to `+1` — a π-rotation, closed to
sub-ppm tolerance:

```
final pulse area θ(T) = 3.1415909 rad (target π)
max |⟨σ_i⟩_numeric − analytic| ≈ 3.4e−06
final ⟨σ_z⟩ = +0.99999999…
```

Wall-clock for the full 400-step time-dependent solve on a 2023
M2 MacBook Air: ~7 ms.

## Physics you can probe next

The envelope is a pure-Python callable, so arbitrary pulse shapes
come for free — no new builder needed. Three natural modifications:

### Blackman window instead of Gaussian

The Blackman window is a compact-support, smoother-edged
alternative to the Gaussian that sits *inside* `[0, T]` with zero
slope at the endpoints. Its cumulative pulse area has a
closed-form expression, but for normalisation you can also just
evaluate it numerically with `scipy.integrate.quad`:

In [ ]:
def blackman_envelope(t: float) -> float:
    if not 0.0 <= t <= pulse_duration_s:
        return 0.0
    x = t / pulse_duration_s
    return (
        0.42
        - 0.5 * math.cos(2 * math.pi * x)
        + 0.08 * math.cos(4 * math.pi * x)
    )

from scipy.integrate import quad
unscaled_area, _ = quad(blackman_envelope, 0.0, pulse_duration_s)
blackman_amplitude = np.pi / (rabi_rad_s * unscaled_area)
print(f"Blackman — unscaled area = {unscaled_area * 1e6:.4f} μs,  "
      f"π-pulse amplitude = {blackman_amplitude:.4f}")

# Compare the two envelope shapes side by side.
t_cmp = np.linspace(0.0, pulse_duration_s, 400)
f_gaussian = np.array([gaussian_envelope(t) for t in t_cmp])
f_blackman  = np.array([blackman_amplitude * blackman_envelope(t) for t in t_cmp])

fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(t_cmp * 1e6, f_gaussian, color=BLUE,  label="Gaussian  (σ = T/10)")
ax.plot(t_cmp * 1e6, f_blackman, color=GREEN, label="Blackman  (compact support)")
ax.set_xlabel("time  (μs)")
ax.set_ylabel("scaled envelope  f(t)")
ax.set_title("Gaussian vs Blackman π-pulse envelope")
ax.legend(frameon=False)
plt.show()
# wrap in a final envelope that multiplies in blackman_amplitude

### Stroboscopic square-wave drive

A square envelope on for a fraction of each mode period — the
`workplan §0.F` benchmark 3 exercises this regime. The envelope
is a simple `1.0` / `0.0` step function; the pulse-area condition
now determines the *duty cycle*, not a continuous amplitude. This
is the closest the modulated-carrier builder gets to emulating a
bang-bang AC drive.

### Adiabatic amplitude ramp

A slowly-varying envelope (e.g. a raised-cosine at both ends plus
a flat middle) is the standard tool for soft turn-on of a drive —
the transient spectral content of a hard step excites unwanted
sidebands when you're running near a mode frequency. This is
purely an envelope change; the rest of the pipeline is unchanged.

## Where to next

- [Tutorial 1](https://uwarring82.github.io/iontrap-dynamics/tutorials/01_first_rabi_readout/) — the static-carrier
  baseline with finite-shot readout.
- [Tutorial 2](https://uwarring82.github.io/iontrap-dynamics/tutorials/02_red_sideband_fock1/) — the sibling swap, but
  to the red-sideband Hamiltonian and Fock `|1⟩` initial state.
- [Phase 1 Architecture](https://uwarring82.github.io/iontrap-dynamics/phase-1-architecture/) — reference
  for the `modulated_carrier_hamiltonian` builder and the full
  list-format dispatch path through `sequences.solve`.
- [`tools/run_demo_gaussian_pulse.py`][demo] — the runnable script
  that produced the plot embedded above; diff it against this
  tutorial for the exact code plus the cumulative-integral
  analytic overlay.

---

## Licence

Sail material — adaptive guidance with specific parameter choices,
not a coastline constraint. Licensed under **CC BY-NC-SA 4.0** per
[`docs/LICENCE`](https://github.com/uwarring82/iontrap-dynamics/blob/main/docs/LICENCE).